In [1]:
BACKEND = "local"   # "local" hoặc "gemini"
N = 100           # số câu hỏi (đặt 5 để chạy thử nhanh)

## 1. Cài đặt và lấy mã

In [2]:
!git clone -q https://github.com/FluteInTheRain/evograph-reproduction.git
%cd evograph-reproduction
!pip install -q -r requirements.txt gdown

/content/evograph-reproduction


## 2. Đăng nhập W&B

In [3]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: khangnguyennhat (khangnguyennhathcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Dữ liệu MetaQA
Thử tải tự động; nếu Google Drive từ chối, hãy tải tay `kb.txt` và `2-hop/vanilla/qa_test.txt` từ https://github.com/yuyuz/MetaQA rồi upload vào thư mục `data/` (đặt tên `kb.txt`, `2hop_qa_test.txt`).

In [4]:
import os
os.makedirs("data", exist_ok=True)
if not (os.path.exists("data/kb.txt") and os.path.exists("data/2hop_qa_test.txt")):
    !gdown --folder https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE -O data/metaqa
    !cp data/metaqa/kb.txt data/kb.txt
    !cp data/metaqa/2-hop/vanilla/qa_test.txt data/2hop_qa_test.txt
!wc -l data/kb.txt data/2hop_qa_test.txt

 134741 data/kb.txt
  14872 data/2hop_qa_test.txt
 149613 total


## 4. Chọn LLM

In [5]:
import subprocess, time, os, requests
if BACKEND == "local":
    !pip install -q vllm
    server = subprocess.Popen(
        ["python", "-m", "vllm.entrypoints.openai.api_server",
         "--model", "Qwen/Qwen2.5-7B-Instruct-AWQ", "--quantization", "awq", "--dtype", "half",
         "--max-model-len", "6144", "--gpu-memory-utilization", "0.9", "--port", "8000"],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            requests.get("http://localhost:8000/v1/models", timeout=2); break
        except Exception:
            time.sleep(5)
    os.environ["LLM_KEY"] = "EMPTY"
    LLM = "--model Qwen/Qwen2.5-7B-Instruct-AWQ --base_url http://localhost:8000/v1 --api_key_env LLM_KEY"
else:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    LLM = "--model gemini-2.5-flash-lite --base_url https://generativelanguage.googleapis.com/v1beta/openai/ --api_key_env GEMINI_API_KEY"
print(LLM)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 60.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━

## 5. Chạy thử nhanh (5 câu)

In [6]:
!python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n 5 --wandb_group smoke-test $LLM

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: khangnguyennhat (khangnguyennhathcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run zvecxbv2 (0.3s)
wandb: ⣯ setting up run zvecxbv2 (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/evograph-reproduction/wandb/run-20261017_080640-zvecxbv2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run full-seed42
wandb: ⭐️ View project at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction
wandb: 🚀 View run at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction/runs/zvecxbv2
wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call

## 6. Bốn cấu hình chính

In [7]:
for v in ["static", "web", "edit", "full"]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --wandb_group main-ablation $LLM

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: khangnguyennhat (khangnguyennhathcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 4o11gwgc (0.3s)
wandb: ⣯ setting up run 4o11gwgc (0.3s)
wandb: ⣟ setting up run 4o11gwgc (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/evograph-reproduction/wandb/run-20261017_080655-4o11gwgc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run static-seed42
wandb: ⭐️ View project at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction
wandb: 🚀 View run at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction/runs/4o11gwgc
wandb: Detected [openai] in use.


## 7. Ablation theo mức khuyết của KG

In [8]:
for d in [0.0, 0.25, 0.5, 0.75]:
    for v in ["static", "full"]:
        !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --drop $d --wandb_group ablation-drop $LLM

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: khangnguyennhat (khangnguyennhathcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 84ex81u8 (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/evograph-reproduction/wandb/run-20261017_081133-84ex81u8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run static-seed42
wandb: ⭐️ View project at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction
wandb: 🚀 View run at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction/runs/84ex81u8
wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip inst

## 8. Ablation theo độ nhiễu của web

In [9]:
for p in [0.0, 0.1, 0.3, 0.5]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n $N --web_noise $p --wandb_group ablation-webnoise $LLM

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: khangnguyennhat (khangnguyennhathcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run zx9rs83y (0.3s)
wandb: ⣯ setting up run zx9rs83y (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/evograph-reproduction/wandb/run-20261017_082013-zx9rs83y
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run full-seed42
wandb: ⭐️ View project at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction
wandb: 🚀 View run at https://wandb.ai/khangnguyennhathcmus/evograph-reproduction/runs/zx9rs83y
wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call

## 9. Tổng hợp kết quả từ W&B
In bảng tóm tắt mọi lần chạy (dán kết quả này vào báo cáo). Nhớ đặt project ở chế độ public.

In [ ]:
import wandb, pandas as pd
api = wandb.Api()
rows = []
for r in api.runs(f"{api.default_entity}/evograph-reproduction"):
    s, c = r.summary, r.config
    if 'hit' not in s or c.get('n') == 5: continue
    rows.append(dict(group=r.group, variant=c['variant'], drop=c['drop'], web_noise=c['web_noise'], n=s.get('n'),
                     hit=round(s['hit'],3), em=round(s.get('em',float('nan')),3), f1=round(s['f1'],3), turns=round(s['avg_turns'],2),
                     edits=round(s['avg_edits'],2), hit_corrupted=round(s['hit_on_corrupted'],3), n_corrupted=s['n_corrupted'], run=r.id))
df = pd.DataFrame(rows).sort_values(['group','drop','web_noise','variant']).reset_index(drop=True)
print(df.to_string())